In [114]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import Dash, html, dash_table, dcc
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import geopandas as gpd

## For my final project, I will be using a dataset from data.gov

#### With this, I will be pasting below the link, as well as the information provided with the dataset.

$\textbf{link}$:
https://catalog.data.gov/dataset/crime-incidents-in-2025

"

The dataset contains a subset of locations and attributes of incidents reported in the ASAP (Analytical Services Application) crime report database by the District of Columbia Metropolitan Police Department (MPD). Visit crimecards.dc.gov for more information. This data is shared via an automated process where addresses are geocoded to the District's Master Address Repository and assigned to the appropriate street block. Block locations for some crime points could not be automatically assigned resulting in 0,0 for x,y coordinates. These can be interactively assigned using the MAR Geocoder.

On February 1 2020, the methodology of geography assignments of crime data was modified to increase accuracy. From January 1 2020 going forward, all crime data will have Ward, ANC, SMD, BID, Neighborhood Cluster, Voting Precinct, Block Group and Census Tract values calculated prior to, rather than after, anonymization to the block level. This change impacts approximately one percent of Ward assignments.



"



This is me typing out my process on what I'll do, but I'm thinking of using tableau to make my final dashboard. With that, I will first be cleaning the data here, renaming the columns that might be confusing, dropping NA / null values so we don't get any skewed results, etc.

This will also help me with understanding the data that I am working with, and will help me think of different visualizations and comparisons I can make with the dataset I have.

In [115]:
data = pd.read_csv('crime.csv')
data.head()

,X,Y,CCN,REPORT_DAT,START_DATE,END_DATE,BLOCK,OFFENSE,METHOD,SHIFT,...,BLOCK_GROUP,CENSUS_TRACT,VOTING_PRECINCT,BID,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID,OCTO_RECORD_ID
0,406451.6900,137312.2500,25136035,2025/09/06 02:05:14+00,2025/09/06 00:28:00+00,2025/09/06 01:43:00+00,5100 - 5199 BLOCK OF SHERIFF ROAD NE,THEFT/OTHER,OTHERS,EVENING,...,007806 1,7806.0,Precinct 93,NaN,406451.690000,137312.250000,38.903642,-76.925620,905044052,NaN
1,404579.2827,137347.1976,25152765,2025/10/06 23:22:30+00,2025/10/06 22:20:00+00,2025/10/06 23:20:00+00,3500 - 3899 BLOCK OF JAY STREET NE,MOTOR VEHICLE THEFT,OTHERS,EVENING,...,009602 2,9602.0,Precinct 100,NaN,404579.282722,137347.197629,38.903969,-76.947206,905044053,NaN
2,402366.6000,133849.5700,25154873,2025/10/11 01:20:36+00,2025/10/10 21:00:00+00,2025/10/10 22:00:00+00,2300 - 2399 BLOCK OF MINNESOTA AVENUE SE,ROBBERY,OTHERS,EVENING,...,007601 1,7601.0,Precinct 133,NaN,402366.600000,133849.570000,38.872470,-76.972728,905044054,NaN
3,397921.3300,138249.9400,25186910,2025/12/12 19:02:29+00,2025/12/11 15:14:00+00,2025/12/11 15:15:00+00,1620 - 1699 BLOCK OF 9TH STREET NW,THEFT/OTHER,OTHERS,DAY,...,004901 3,4901.0,Precinct 21,NaN,397921.330000,138249.940000,38.912111,-77.023967,905044058,NaN
4,394625.5600,137287.0500,25066070,2025/05/06 02:55:58+00,2025/05/05 17:18:00+00,2025/05/05 19:36:00+00,3100 - 3199 BLOCK OF SOUTH STREET NW,THEFT F/AUTO,OTHERS,EVENING,...,000102 3,102.0,Precinct 5,GEORGETOWN,394625.560000,137287.050000,38.903422,-77.061961,905044060,NaN


In [116]:
data.shape

(24152, 25)

In [117]:
data.columns

Index(['X', 'Y', 'CCN', 'REPORT_DAT', 'START_DATE', 'END_DATE', 'BLOCK',
       'OFFENSE', 'METHOD', 'SHIFT', 'WARD', 'ANC', 'DISTRICT', 'PSA',
       'NEIGHBORHOOD_CLUSTER', 'BLOCK_GROUP', 'CENSUS_TRACT',
       'VOTING_PRECINCT', 'BID', 'XBLOCK', 'YBLOCK', 'LATITUDE', 'LONGITUDE',
       'OBJECTID', 'OCTO_RECORD_ID'],
      dtype='object')

In [118]:
crime = data.copy() # making a copy so we can edit our new dataset

In [119]:
crime = crime.rename(columns = 
            {'X': 'x',
             'Y': 'y',
             'REPORT_DAT': 'report date',
             'START_DATE': 'start time',
             'END_DATE': 'end date',
             'OFFENSE': 'crime',
             'METHOD': 'method',
             'SHIFT': 'time',
             'VOTING_PRECINCT': 'precint'}).drop(columns = ['OCTO_RECORD_ID', 'BID'])
crime.head()

,x,y,CCN,report date,start time,end date,BLOCK,crime,method,time,...,PSA,NEIGHBORHOOD_CLUSTER,BLOCK_GROUP,CENSUS_TRACT,precint,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID
0,406451.6900,137312.2500,25136035,2025/09/06 02:05:14+00,2025/09/06 00:28:00+00,2025/09/06 01:43:00+00,5100 - 5199 BLOCK OF SHERIFF ROAD NE,THEFT/OTHER,OTHERS,EVENING,...,602.0,Cluster 31,007806 1,7806.0,Precinct 93,406451.690000,137312.250000,38.903642,-76.925620,905044052
1,404579.2827,137347.1976,25152765,2025/10/06 23:22:30+00,2025/10/06 22:20:00+00,2025/10/06 23:20:00+00,3500 - 3899 BLOCK OF JAY STREET NE,MOTOR VEHICLE THEFT,OTHERS,EVENING,...,601.0,Cluster 30,009602 2,9602.0,Precinct 100,404579.282722,137347.197629,38.903969,-76.947206,905044053
2,402366.6000,133849.5700,25154873,2025/10/11 01:20:36+00,2025/10/10 21:00:00+00,2025/10/10 22:00:00+00,2300 - 2399 BLOCK OF MINNESOTA AVENUE SE,ROBBERY,OTHERS,EVENING,...,607.0,Cluster 34,007601 1,7601.0,Precinct 133,402366.600000,133849.570000,38.872470,-76.972728,905044054
3,397921.3300,138249.9400,25186910,2025/12/12 19:02:29+00,2025/12/11 15:14:00+00,2025/12/11 15:15:00+00,1620 - 1699 BLOCK OF 9TH STREET NW,THEFT/OTHER,OTHERS,DAY,...,307.0,Cluster 7,004901 3,4901.0,Precinct 21,397921.330000,138249.940000,38.912111,-77.023967,905044058
4,394625.5600,137287.0500,25066070,2025/05/06 02:55:58+00,2025/05/05 17:18:00+00,2025/05/05 19:36:00+00,3100 - 3199 BLOCK OF SOUTH STREET NW,THEFT F/AUTO,OTHERS,EVENING,...,206.0,Cluster 4,000102 3,102.0,Precinct 5,394625.560000,137287.050000,38.903422,-77.061961,905044060


In [120]:
crime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24152 entries, 0 to 24151
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   x                     24152 non-null  float64
 1   y                     24152 non-null  float64
 2   CCN                   24152 non-null  int64  
 3   report date           24152 non-null  object 
 4   start time            24152 non-null  object 
 5   end date              22459 non-null  object 
 6   BLOCK                 24152 non-null  object 
 7   crime                 24152 non-null  object 
 8   method                24152 non-null  object 
 9   time                  24152 non-null  object 
 10  WARD                  24152 non-null  int64  
 11  ANC                   24152 non-null  object 
 12  DISTRICT              24120 non-null  float64
 13  PSA                   24120 non-null  float64
 14  NEIGHBORHOOD_CLUSTER  24152 non-null  object 
 15  BLOCK_GROUP        

In [121]:
# 3   report date           24152 non-null  object 
# 4   start time            24152 non-null  object 
# 5   end date              22459 non-null  object 
# want to convert these into time 

crime['report date'] = pd.to_datetime(crime['report date'])
crime['start time'] = pd.to_datetime(crime['start time'])
crime['end date'] = pd.to_datetime(crime['end date'])

In [122]:
# here, we are going to make a new column for just months, so we have more data to compare with
# when we start putting it on tableau
# i will also make a time of day (hour) column so we can compare at what times of day is crime higher

In [123]:
crime['month'] = crime['report date'].dt.month
crime['hour'] = crime['start time'].dt.hour

In [124]:
crime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24152 entries, 0 to 24151
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   x                     24152 non-null  float64            
 1   y                     24152 non-null  float64            
 2   CCN                   24152 non-null  int64              
 3   report date           24152 non-null  datetime64[ns, UTC]
 4   start time            24152 non-null  datetime64[ns, UTC]
 5   end date              22459 non-null  datetime64[ns, UTC]
 6   BLOCK                 24152 non-null  object             
 7   crime                 24152 non-null  object             
 8   method                24152 non-null  object             
 9   time                  24152 non-null  object             
 10  WARD                  24152 non-null  int64              
 11  ANC                   24152 non-null  object             
 12  DIST

In [125]:
crime.head()

,x,y,CCN,report date,start time,end date,BLOCK,crime,method,time,...,BLOCK_GROUP,CENSUS_TRACT,precint,XBLOCK,YBLOCK,LATITUDE,LONGITUDE,OBJECTID,month,hour
0,406451.6900,137312.2500,25136035,2025-09-06 02:05:14+00:00,2025-09-06 00:28:00+00:00,2025-09-06 01:43:00+00:00,5100 - 5199 BLOCK OF SHERIFF ROAD NE,THEFT/OTHER,OTHERS,EVENING,...,007806 1,7806.0,Precinct 93,406451.690000,137312.250000,38.903642,-76.925620,905044052,9,0
1,404579.2827,137347.1976,25152765,2025-10-06 23:22:30+00:00,2025-10-06 22:20:00+00:00,2025-10-06 23:20:00+00:00,3500 - 3899 BLOCK OF JAY STREET NE,MOTOR VEHICLE THEFT,OTHERS,EVENING,...,009602 2,9602.0,Precinct 100,404579.282722,137347.197629,38.903969,-76.947206,905044053,10,22
2,402366.6000,133849.5700,25154873,2025-10-11 01:20:36+00:00,2025-10-10 21:00:00+00:00,2025-10-10 22:00:00+00:00,2300 - 2399 BLOCK OF MINNESOTA AVENUE SE,ROBBERY,OTHERS,EVENING,...,007601 1,7601.0,Precinct 133,402366.600000,133849.570000,38.872470,-76.972728,905044054,10,21
3,397921.3300,138249.9400,25186910,2025-12-12 19:02:29+00:00,2025-12-11 15:14:00+00:00,2025-12-11 15:15:00+00:00,1620 - 1699 BLOCK OF 9TH STREET NW,THEFT/OTHER,OTHERS,DAY,...,004901 3,4901.0,Precinct 21,397921.330000,138249.940000,38.912111,-77.023967,905044058,12,15
4,394625.5600,137287.0500,25066070,2025-05-06 02:55:58+00:00,2025-05-05 17:18:00+00:00,2025-05-05 19:36:00+00:00,3100 - 3199 BLOCK OF SOUTH STREET NW,THEFT F/AUTO,OTHERS,EVENING,...,000102 3,102.0,Precinct 5,394625.560000,137287.050000,38.903422,-77.061961,905044060,5,17


In [126]:
crime[['LATITUDE','LONGITUDE']].isna().sum()

LATITUDE     0
LONGITUDE    0
dtype: int64

In [128]:
crime.columns

Index(['x', 'y', 'CCN', 'report date', 'start time', 'end date', 'BLOCK',
       'crime', 'method', 'time', 'WARD', 'ANC', 'DISTRICT', 'PSA',
       'NEIGHBORHOOD_CLUSTER', 'BLOCK_GROUP', 'CENSUS_TRACT', 'precint',
       'XBLOCK', 'YBLOCK', 'LATITUDE', 'LONGITUDE', 'OBJECTID', 'month',
       'hour'],
      dtype='object')

In [130]:
# after checking, i found that XBLOCK & YBLOCK are essentially the same as x & y, so i will drop them

In [131]:
crime = crime.drop(columns = ['XBLOCK', 'YBLOCK'])

In [133]:
crime.head()

,x,y,CCN,report date,start time,end date,BLOCK,crime,method,time,...,PSA,NEIGHBORHOOD_CLUSTER,BLOCK_GROUP,CENSUS_TRACT,precint,LATITUDE,LONGITUDE,OBJECTID,month,hour
0,406451.6900,137312.2500,25136035,2025-09-06 02:05:14+00:00,2025-09-06 00:28:00+00:00,2025-09-06 01:43:00+00:00,5100 - 5199 BLOCK OF SHERIFF ROAD NE,THEFT/OTHER,OTHERS,EVENING,...,602.0,Cluster 31,007806 1,7806.0,Precinct 93,38.903642,-76.925620,905044052,9,0
1,404579.2827,137347.1976,25152765,2025-10-06 23:22:30+00:00,2025-10-06 22:20:00+00:00,2025-10-06 23:20:00+00:00,3500 - 3899 BLOCK OF JAY STREET NE,MOTOR VEHICLE THEFT,OTHERS,EVENING,...,601.0,Cluster 30,009602 2,9602.0,Precinct 100,38.903969,-76.947206,905044053,10,22
2,402366.6000,133849.5700,25154873,2025-10-11 01:20:36+00:00,2025-10-10 21:00:00+00:00,2025-10-10 22:00:00+00:00,2300 - 2399 BLOCK OF MINNESOTA AVENUE SE,ROBBERY,OTHERS,EVENING,...,607.0,Cluster 34,007601 1,7601.0,Precinct 133,38.872470,-76.972728,905044054,10,21
3,397921.3300,138249.9400,25186910,2025-12-12 19:02:29+00:00,2025-12-11 15:14:00+00:00,2025-12-11 15:15:00+00:00,1620 - 1699 BLOCK OF 9TH STREET NW,THEFT/OTHER,OTHERS,DAY,...,307.0,Cluster 7,004901 3,4901.0,Precinct 21,38.912111,-77.023967,905044058,12,15
4,394625.5600,137287.0500,25066070,2025-05-06 02:55:58+00:00,2025-05-05 17:18:00+00:00,2025-05-05 19:36:00+00:00,3100 - 3199 BLOCK OF SOUTH STREET NW,THEFT F/AUTO,OTHERS,EVENING,...,206.0,Cluster 4,000102 3,102.0,Precinct 5,38.903422,-77.061961,905044060,5,17


In [135]:
# now ill save this one into a new csv file and use that for tableau

crime.to_csv('cleaned_crimes.csv', index = False)

Link to the workbook:

https://public.tableau.com/views/DSCI410LFINALPROJECT/Dashboard1?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link